# 01 — Data Acquisition, EDA & Dataset Construction
**Phase 1** of SafeTails. The species classifier is the *only* in-house trained model.

Public, reproducible sources (see `ml/src/acquire_data.py`): Cat/Dog from `Bingsu/Cat_and_Dog`; Cow=ImageNet *ox*, Buffalo=ImageNet *water buffalo*; Other = a mix of ImageNet animal synsets. `SEED=42` throughout.

> **v2 update:** the *deployed* model is now **ConvNeXt-Tiny (98.4% acc)**, retrained on an upgraded pipeline (Oxford-IIIT Pet + Stanford Dogs + ImageNet, RandAugment/MixUp/CutMix, Optuna, calibration) in **`03_colab_train_species.ipynb`** (Colab Pro). Notebooks 01-02 document the Phase-2 baseline (v1 EfficientNet-B0, 94.4%) kept for the comparison narrative.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import os; os.chdir(ROOT)
print('repo root:', ROOT)

## 1. Acquisition
Run once (downloads into `data/raw/<Class>/`):
```bash
python -m ml.src.acquire_data --per-class 400
```

In [ ]:
from ml.src import pipeline as P
from ml.src.config import CLASS_NAMES
df = P.scan_raw()
print('total images:', len(df))
df.groupby('label').size().reindex(CLASS_NAMES)

## 2. EDA — class balance, image sizes, sample grid

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
counts = df.groupby('label').size().reindex(CLASS_NAMES)
counts.plot(kind='bar', color='#157d8f', title='Images per class'); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
samp = df.sample(min(500, len(df)), random_state=42)
ws, hs, ars, brs = [], [], [], []
for p in samp['path']:
    try:
        with Image.open(p) as im:
            im = im.convert('RGB'); ws.append(im.width); hs.append(im.height)
            ars.append(im.width/max(im.height,1))
            brs.append(np.asarray(im.resize((64,64))).mean()/255)
    except Exception: pass
fig, ax = plt.subplots(2,2, figsize=(9,6))
ax[0,0].hist(ws, bins=30, color='#3aa3b5'); ax[0,0].set_title('width (px)')
ax[0,1].hist(hs, bins=30, color='#d98a1f'); ax[0,1].set_title('height (px)')
ax[1,0].hist(ars, bins=30, color='#6d5bd0'); ax[1,0].set_title('aspect ratio (w/h)')
ax[1,1].hist(brs, bins=30, color='#3f7ec2'); ax[1,1].set_title('mean brightness (0-1)')
plt.tight_layout(); plt.show()
print('median size:', int(np.median(ws)), 'x', int(np.median(hs)),
      '| median aspect:', round(float(np.median(ars)),2),
      '| median brightness:', round(float(np.median(brs)),2))

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 5, figsize=(11, 2.1*len(CLASS_NAMES)))
for r, cls in enumerate(CLASS_NAMES):
    rows = df[df['label']==cls]['path'].head(5).tolist()
    for c in range(5):
        ax = axes[r][c]; ax.axis('off')
        if c < len(rows):
            ax.imshow(Image.open(rows[c]).convert('RGB').resize((128,128)))
        if c==0: ax.set_title(cls, loc='left', fontsize=11)
plt.tight_layout(); plt.show()

## 3. Duplicate check (perceptual hash)
The same pHash guard the anti-spam system uses, applied to the dataset.

In [ ]:
import imagehash
seen, dups = {}, 0
for p in samp['path']:
    try:
        h = str(imagehash.phash(Image.open(p)))
        dups += 1 if h in seen else 0; seen[h]=p
    except Exception: pass
print(f'near-duplicate hashes in sample: {dups}')

## 4. Stratified 70/15/15 split → `data/splits.csv`

In [ ]:
m = P.build_split_manifest()
print(m['split'].value_counts().to_dict())
m.groupby(['label','split']).size().unstack(fill_value=0).reindex(CLASS_NAMES)

## 5. Augmentation preview (mimics noisy phone photos)

In [ ]:
train_tf, eval_tf = P.build_transforms(augment=True)
import torch
inv = lambda t: (t*torch.tensor(P._STD)[:,None,None]+torch.tensor(P._MEAN)[:,None,None]).clamp(0,1).permute(1,2,0).numpy()
path = df.iloc[0]['path']; img = Image.open(path).convert('RGB')
fig, ax = plt.subplots(1,5, figsize=(11,2.4))
for i in range(5): ax[i].imshow(inv(train_tf(img))); ax[i].axis('off')
plt.suptitle('5 random augmentations'); plt.tight_layout(); plt.show()

**Next →** `02_species_classification.ipynb` (train, compare, calibrate, export).